In [1]:
import zipfile
import os
import glob
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import warnings

# Load data


In [2]:
base_path = Path(os.getcwd())

CSV_PATH  = base_path / r"..\..\Data\Raw\GME_MGP_NORD_2005_2026.csv"

df_raw = pd.read_csv(CSV_PATH, parse_dates=["datetime", "date"])

In [3]:
print(f"Raw dataset shape : {df_raw.shape}")
print(f"Date range        : {df_raw['datetime'].min()}  →  {df_raw['datetime'].max()}")

Raw dataset shape : (186003, 6)
Date range        : 2005-01-01 00:00:00  →  2026-03-22 23:00:00


In [4]:
# Build a full continuous hourly index and reindex — this materialises all gaps
full_index = pd.date_range(
    start = df_raw["datetime"].min(),
    end   = df_raw["datetime"].max(),
    freq  = "h"
)
 
df = df_raw.set_index("datetime").reindex(full_index)
df.index.name = "datetime"
 
# Count gaps introduced by reindex
n_gaps = df["price_NORD_EURMWh"].isna().sum()
print(f"\n[STEP 1] Complete hourly index built.")
print(f"  Gaps materialised (missing hours) : {n_gaps}")
 
# Show the gap timestamps for inspection
gap_timestamps = df[df["price_NORD_EURMWh"].isna()].index
print(f"  Gap timestamps:")
for ts in gap_timestamps:
    print(f"    {ts}")
 


[STEP 1] Complete hourly index built.
  Gaps materialised (missing hours) : 21
  Gap timestamps:
    2005-03-27 23:00:00
    2006-03-26 23:00:00
    2007-03-25 23:00:00
    2008-03-30 23:00:00
    2009-03-29 23:00:00
    2010-03-28 23:00:00
    2011-03-27 23:00:00
    2012-03-25 23:00:00
    2013-03-31 23:00:00
    2014-03-30 23:00:00
    2015-03-29 23:00:00
    2016-03-27 23:00:00
    2017-03-26 23:00:00
    2018-03-25 23:00:00
    2019-03-31 23:00:00
    2020-03-29 23:00:00
    2021-03-28 23:00:00
    2022-03-27 23:00:00
    2023-03-26 23:00:00
    2024-03-31 23:00:00
    2025-03-30 23:00:00


In [5]:
zeros = df[df["price_NORD_EURMWh"] == 0].copy()
zeros["year"] = zeros.index.year
 
print(f"\n[STEP 2] Zero price investigation")
print(f"  Total zero prices : {len(zeros)}")
print(f"\n  Distribution by year:")
print(zeros.groupby("year").size().to_string())
print(f"\n  Distribution by hour:")
print(zeros.groupby(zeros.index.hour).size().to_string())
 
# Flag zeros for treatment decision
# Zeros pre-2010: likely parsing artifacts → treat as NaN
# Zeros post-2010: plausible market outcome → flag but keep
ZERO_ARTIFACT_CUTOFF = 2010
df["zero_flag"] = (df["price_NORD_EURMWh"] == 0)
 
n_zero_artifact = ((df["price_NORD_EURMWh"] == 0) & (df.index.year <= ZERO_ARTIFACT_CUTOFF)).sum()
n_zero_valid    = ((df["price_NORD_EURMWh"] == 0) & (df.index.year >  ZERO_ARTIFACT_CUTOFF)).sum()
print(f"\n  Zeros ≤ {ZERO_ARTIFACT_CUTOFF} (likely artifacts) : {n_zero_artifact}")
print(f"  Zeros >  {ZERO_ARTIFACT_CUTOFF} (plausible market) : {n_zero_valid}")
 
# Convert artifact zeros to NaN
df.loc[(df["price_NORD_EURMWh"] == 0) & (df.index.year <= ZERO_ARTIFACT_CUTOFF),
       "price_NORD_EURMWh"] = np.nan


[STEP 2] Zero price investigation
  Total zero prices : 33

  Distribution by year:
year
2005    14
2013     4
2020     5
2025    10

  Distribution by hour:
datetime
0      1
1      1
2      1
3      1
4      1
5      1
7      1
9      1
10     1
11     2
12     1
13     5
14    10
15     4
16     1
22     1

  Zeros ≤ 2010 (likely artifacts) : 14
  Zeros >  2010 (plausible market) : 19


In [6]:
extremes = df[df["price_NORD_EURMWh"] > 500].copy()
extremes["year"] = extremes.index.year
 
print(f"\n[STEP 3] Extreme price investigation (> 500 €/MWh)")
print(f"  Total extreme records : {len(extremes)}")
print(f"\n  Distribution by year:")
print(extremes.groupby("year").size().to_string())
print(f"\n  Distribution by hour:")
print(extremes.groupby(extremes.index.hour).size().to_string())
print(f"\n  Max price record:")
print(df.loc[df["price_NORD_EURMWh"].idxmax(), ["price_NORD_EURMWh"]].to_string())
print(f"  At datetime: {df['price_NORD_EURMWh'].idxmax()}")
 
# Flag extreme prices — do NOT remove, they are real market events
# Use a regime flag instead: 0=normal, 1=spike (>P95 by year), 2=extreme (>500)
df["price_flag"] = 0
df.loc[df["price_NORD_EURMWh"] > 500, "price_flag"] = 2


[STEP 3] Extreme price investigation (> 500 €/MWh)
  Total extreme records : 891

  Distribution by year:
year
2021     13
2022    878

  Distribution by hour:
datetime
0     26
1     14
2     14
3     13
4     13
5     15
6     26
7     41
8     60
9     43
10    38
11    36
12    31
13    30
14    32
15    34
16    37
17    51
18    59
19    79
20    72
21    50
22    43
23    34

  Max price record:
price_NORD_EURMWh    871.0
  At datetime: 2022-08-29 19:00:00


# Filling missing values

In [7]:
# Strategy:
#   - Isolated gaps (1 hour): linear interpolation
#   - PUN NaN: forward-fill (max 3 hours)
#   - All remaining NaN after interpolation: forward-fill safety net
 
n_nan_before = df["price_NORD_EURMWh"].isna().sum()
 
# Linear interpolation for NORD (handles DST gaps and artifact zeros converted to NaN)
df["price_NORD_EURMWh"] = df["price_NORD_EURMWh"].interpolate(
    method="linear", limit=3, limit_direction="both"
)
 
# Flag all imputed records
df["imputed_flag"] = False
df.loc[df_raw.set_index("datetime").reindex(full_index)["price_NORD_EURMWh"].isna(),
       "imputed_flag"] = True
 
# PUN: forward-fill (secondary variable, not primary)
df["price_PUN_EURMWh"] = df["price_PUN_EURMWh"].ffill(limit=3)
 
n_nan_after = df["price_NORD_EURMWh"].isna().sum()
print(f"\n[STEP 4] Imputation complete.")
print(f"  NORD NaN before : {n_nan_before}")
print(f"  NORD NaN after  : {n_nan_after}")
print(f"  Records imputed : {df['imputed_flag'].sum()}")
 


[STEP 4] Imputation complete.
  NORD NaN before : 35
  NORD NaN after  : 0
  Records imputed : 21


In [8]:
df["p95_year"] = df.groupby(df.index.year)["price_NORD_EURMWh"].transform(
    lambda x: x.quantile(0.95)
)
df.loc[
    (df["price_NORD_EURMWh"] > df["p95_year"]) & (df["price_flag"] == 0),
    "price_flag"
] = 1
 
spike_counts = df.groupby("price_flag").size()
print(f"\n[STEP 5] Price regime flags assigned.")
print(f"  0 = normal   : {spike_counts.get(0,0):>10,}")
print(f"  1 = spike     : {spike_counts.get(1,0):>10,}  (above P95 within year)")
print(f"  2 = extreme   : {spike_counts.get(2,0):>10,}  (above 500 €/MWh)")
 
 


[STEP 5] Price regime flags assigned.
  0 = normal   :    176,310
  1 = spike     :      8,823  (above P95 within year)
  2 = extreme   :        891  (above 500 €/MWh)


# Feature Engineering

In [9]:
df["year"]       = df.index.year
df["month"]      = df.index.month
df["day"]        = df.index.day
df["hour"]       = df.index.hour
df["weekday"]    = df.index.dayofweek        # 0=Mon, 6=Sun
df["quarter"]    = df.index.quarter
df["week"]       = df.index.isocalendar().week.astype(int)
df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)
df["is_peak"]    = df["hour"].between(8, 20).astype(int)

In [10]:
df["log_price"]    = np.log(df["price_NORD_EURMWh"].clip(lower=0.01))
df["price_lag1h"]  = df["price_NORD_EURMWh"].shift(1)
df["price_lag24h"] = df["price_NORD_EURMWh"].shift(24)
df["price_lag168h"]= df["price_NORD_EURMWh"].shift(168)   # 1 week
 
# Hourly log return
df["log_return_1h"] = np.log(
    df["price_NORD_EURMWh"].clip(lower=0.01) /
    df["price_NORD_EURMWh"].clip(lower=0.01).shift(1)
)

In [11]:
# ── Rolling statistics ─────────────────────────────────────────────────────────

df["rolling_mean_7d"]  = df["price_NORD_EURMWh"].rolling(168,  min_periods=1).mean()
df["rolling_mean_30d"] = df["price_NORD_EURMWh"].rolling(720,  min_periods=1).mean()
df["rolling_std_7d"]   = df["price_NORD_EURMWh"].rolling(168,  min_periods=24).std()
df["rolling_std_30d"]  = df["price_NORD_EURMWh"].rolling(720,  min_periods=168).std()

In [12]:
# ── Intraday spread (daily max - min) ──────────────────────────────────────────
daily_stats = df.groupby(df.index.date)["price_NORD_EURMWh"].agg(
    daily_max="max", daily_min="min", daily_mean="mean"
)
daily_stats.index = pd.to_datetime(daily_stats.index)
daily_stats["daily_spread"] = daily_stats["daily_max"] - daily_stats["daily_min"]
 
df["daily_max"]    = df.index.normalize().map(daily_stats["daily_max"])
df["daily_min"]    = df.index.normalize().map(daily_stats["daily_min"])
df["daily_mean"]   = df.index.normalize().map(daily_stats["daily_mean"])
df["daily_spread"] = df.index.normalize().map(daily_stats["daily_spread"])
 

In [13]:
# ── Price deviation from daily mean (intraday shape) ──────────────────────────
df["price_vs_daily_mean"] = df["price_NORD_EURMWh"] - df["daily_mean"]
 

In [14]:
# ── Monthly climatological baseline ───────────────────────────────────────────
monthly_clim = df.groupby(["month", "hour"])["price_NORD_EURMWh"].mean()
monthly_clim.name = "clim_mean_month_hour"
df = df.join(monthly_clim, on=["month", "hour"])
 
df["price_anomaly"] = df["price_NORD_EURMWh"] - df["clim_mean_month_hour"]
 

In [15]:
# ── BESS arbitrage proxy: next 24h max - current price ────────────────────────
df["fwd_max_24h"]       = df["price_NORD_EURMWh"].rolling(24).max().shift(-24)
df["bess_arb_proxy"]    = (df["fwd_max_24h"] - df["price_NORD_EURMWh"]).clip(lower=0)
 
print("  Features engineered:")
new_cols = ["log_price","price_lag1h","price_lag24h","price_lag168h",
            "log_return_1h","rolling_mean_7d","rolling_mean_30d",
            "rolling_std_7d","rolling_std_30d","daily_spread",
            "price_vs_daily_mean","price_anomaly","bess_arb_proxy"]
for c in new_cols:
    print(f"    + {c}")
 
 

  Features engineered:
    + log_price
    + price_lag1h
    + price_lag24h
    + price_lag168h
    + log_return_1h
    + rolling_mean_7d
    + rolling_mean_30d
    + rolling_std_7d
    + rolling_std_30d
    + daily_spread
    + price_vs_daily_mean
    + price_anomaly
    + bess_arb_proxy


In [16]:
OUTPUT_CLEAN = base_path / r"..\..\Data\Cleaned\df_energy_cleaned.parquet"

df.reset_index().to_parquet(OUTPUT_CLEAN, index=False)
print(f"\n[SAVED] {OUTPUT_CLEAN}")
print(f"  Final shape: {df.shape}")


[SAVED] c:\Users\LucasMonero\Documents\data projects\Master Thesis\Project\Code\Transformation\..\..\Data\Cleaned\df_energy_cleaned.parquet
  Final shape: (186024, 35)


In [17]:
df.columns

Index(['date', 'hour', 'price_NORD_EURMWh', 'price_PUN_EURMWh', 'mercato',
       'zero_flag', 'price_flag', 'imputed_flag', 'p95_year', 'year', 'month',
       'day', 'weekday', 'quarter', 'week', 'is_weekend', 'is_peak',
       'log_price', 'price_lag1h', 'price_lag24h', 'price_lag168h',
       'log_return_1h', 'rolling_mean_7d', 'rolling_mean_30d',
       'rolling_std_7d', 'rolling_std_30d', 'daily_max', 'daily_min',
       'daily_mean', 'daily_spread', 'price_vs_daily_mean',
       'clim_mean_month_hour', 'price_anomaly', 'fwd_max_24h',
       'bess_arb_proxy'],
      dtype='str')